# Data Acquisition Guide
## National Transport Decarbonisation Dashboard for Ireland

This notebook is the **single reference point** for every data source used in the
project. It documents where each dataset comes from, how to download or refresh
it, and how to name and store it so the rest of the pipeline picks it up
automatically.

> **Run this notebook first** whenever you need to refresh the data. After
> downloading, run `01_data_preprocessing_and_cleaning.ipynb` to rebuild the
> processed outputs.

### Principle: single source of truth
All raw data must come from the official CSO PxStat / data.gov.ie portals. No
ad-hoc alternative estimates or copy-pasted figures. This ensures every KPI and
forecast in the dashboard is fully traceable back to a citable official source.

## 1. Data source catalogue

The table below documents all eight datasets used in the project, plus the
supplementary NaPTAN spatial layer.

In [1]:
import os
from pathlib import Path

# Walk up from wherever Jupyter started until we find the repo root
def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = Path(os.environ.get("TDD_REPO_ROOT", str(_find_repo_root())))
RAW_DIR   = Path(os.environ.get("TDD_RAW_DIR",   str(REPO_ROOT / "data" / "raw")))
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DOCS_DIR      = REPO_ROOT / "docs"

print("Root Repo :", REPO_ROOT)
print("Raw directory   :", RAW_DIR)

if RAW_DIR.exists():
    print("\nDirectory Exists")
else:
    print("\nDirectory Missing")

Root Repo : /Users/ganeshsundararaman.arumugam/Desktop/Transport_latest/Transport_Decarbonisation_Dashboard_IE
Raw directory   : /Users/ganeshsundararaman.arumugam/Desktop/Transport_latest/Transport_Decarbonisation_Dashboard_IE/data/raw

Directory Exists


In [2]:
import os, glob
from pathlib import Path
from datetime import datetime

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = Path(os.environ.get("TDD_RAW_DIR", REPO_ROOT / "data" / "raw"))

CATALOGUE = [
    {
        "id":          "TOA11",
        "description": "Luas Passenger Numbers (Red & Green line, monthly)",
        "source":      "CSO PxStat / data.gov.ie",
        "time_range":  "2018–2025",
        "frequency":   "Monthly (released ~6 weeks after reference month)",
        "url":         "https://data.cso.ie/table/TOA11",
        "use":         "Public Transport Usage Index (Luas journeys per capita)",
    },
    {
        "id":          "PEA01",
        "description": "Population Estimates by Age Group and Sex (April reference)",
        "source":      "CSO PxStat",
        "time_range":  "2018–2025",
        "frequency":   "Annual (released ~August each year)",
        "url":         "https://data.cso.ie/table/PEA01",
        "use":         "Denominator for all per-capita KPIs",
    },
    {
        "id":          "THA25",
        "description": "Public Transport Passenger Journeys by Mode (weekly, bus & rail, excl. Luas)",
        "source":      "CSO PxStat / data.gov.ie",
        "time_range":  "2019–2025",
        "frequency":   "Weekly (released ~2 weeks in arrears)",
        "url":         "https://data.cso.ie/table/THA25",
        "use":         "Public Transport Usage Index (bus + rail journeys)",
    },
    {
        "id":          "THA18",
        "description": "Road Traffic – Private Cars by Engine Capacity, Fuel & County",
        "source":      "CSO PxStat",
        "time_range":  "2018–2023",
        "frequency":   "Annual (released ~Q2 following reference year)",
        "url":         "https://data.cso.ie/table/THA18",
        "use":         "Car Dependency Index; private-car km for intensity cross-check",
    },
    {
        "id":          "THA17",
        "description": "Road Traffic Volumes – All Vehicles by Fuel Type & County",
        "source":      "CSO PxStat",
        "time_range":  "2018–2023",
        "frequency":   "Annual (released ~Q2 following reference year)",
        "url":         "https://data.cso.ie/table/THA17",
        "use":         "Transport Intensity Indicator (total vehicle-km per capita)",
    },
    {
        "id":          "TEM12",
        "description": "New Vehicles Licensed for the First Time by Fuel Type (monthly)",
        "source":      "CSO PxStat",
        "time_range":  "2015–2026",
        "frequency":   "Monthly (released within ~2 weeks of month end)",
        "url":         "https://data.cso.ie/table/TEM12",
        "use":         "Fuel Transition analysis (petrol → EV/hybrid share trend)",
    },
    {
        "id":          "TEM22",
        "description": "New & Second-hand Private Cars Licensed (historical 2019–2021)",
        "source":      "CSO PxStat",
        "time_range":  "2019–2021",
        "frequency":   "Superseded by TEM23 — static historical archive",
        "url":         "https://data.cso.ie/table/TEM22",
        "use":         "Private car registration series 2019–2021 (bridged to TEM23)",
    },
    {
        "id":          "TEM23",
        "description": "All Private Cars Licensed for the First Time (new + imported)",
        "source":      "CSO PxStat",
        "time_range":  "2022–2026",
        "frequency":   "Monthly (released within ~2 weeks of month end)",
        "url":         "https://data.cso.ie/table/TEM23",
        "use":         "Private car registration series 2022–2026",
    },
    {
        "id":          "NaPTAN",
        "description": "National Public Transport Access Nodes – Bus Stop Points",
        "source":      "data.gov.ie (National Transport Authority)",
        "time_range":  "Static (updated periodically)",
        "frequency":   "No fixed schedule — check data.gov.ie for updates",
        "url":         "https://data.gov.ie/dataset/national-public-transport-access-nodes-naptan",
        "use":         "Optional map layer showing bus/rail stop coverage",
    },
]

print(f"{'Table':<7}  {'Description':<50}  {'Range':<12}  {'Frequency'}")
print("-" * 90)
for row in CATALOGUE:
    desc = row["description"][:50]
    print(f"{row['id']:<7}  {desc:<50}  {row['time_range']:<12}  {row['frequency']}")

Table    Description                                         Range         Frequency
------------------------------------------------------------------------------------------
TOA11    Luas Passenger Numbers (Red & Green line, monthly)  2018–2025     Monthly (released ~6 weeks after reference month)
PEA01    Population Estimates by Age Group and Sex (April r  2018–2025     Annual (released ~August each year)
THA25    Public Transport Passenger Journeys by Mode (weekl  2019–2025     Weekly (released ~2 weeks in arrears)
THA18    Road Traffic – Private Cars by Engine Capacity, Fu  2018–2023     Annual (released ~Q2 following reference year)
THA17    Road Traffic Volumes – All Vehicles by Fuel Type &  2018–2023     Annual (released ~Q2 following reference year)
TEM12    New Vehicles Licensed for the First Time by Fuel T  2015–2026     Monthly (released within ~2 weeks of month end)
TEM22    New & Second-hand Private Cars Licensed (historica  2019–2021     Superseded by TEM23 — static hist

## 2. Manual download

Go to the table page on CSO PxStat (e.g. data.cso.ie/table/TEM23),
make sure the full date range is selected, hit Download -> CSV, and drop it in data/raw/.

## File naming convention

```
<TABLE_ID>_<YYYYMMDDTHHMMSS>_<STARTYEAR><ENDYEAR>.csv
```

Examples already in `data/raw/`:
```
TEM23_20260528T000533_20222026.csv
TOA11_20260528T000523_20182025.csv
```

The pipeline matches files by **prefix only** (`TEM23*.csv`), so the timestamp
and year suffix are metadata-for-humans, not parsed by code. Newer downloads
with a later timestamp are picked up automatically — old copies can be deleted
or archived.

Don't touch the column headers - if CSO changes them, fix it in notebook 01,
not here.

## 3. Programmatic download via the CSO PxStat API

The CSO PxStat API uses the **JSON-RPC 2.0** protocol. No API key is required.
The endpoint supports CSV, JSON-stat and PX format responses.

Use this approach if you need to automate refreshes on a schedule (e.g. in a
GitHub Actions workflow). For ad-hoc updates, the manual method in section 2 is
simpler.

In [3]:
import requests
from datetime import datetime, timezone
import re

API_ENDPOINT = "https://ws.cso.ie/public/api.jsonrpc"

def download_table(table_id, raw_dir=RAW_DIR):
    payload = {
        "jsonrpc": "2.0",
        "method": "PxStat.Data.Cube_API.ReadDataset",
        "params": {
            "class": "query", "id": [], "dimension": {},
            "extension": {"pivot": None, "codes": False,
                          "language": {"code": "en"},
                          "format": {"type": "CSV", "version": "2.0"},
                          "matrix": table_id.upper()},
            "version": "2.0",
        },
    }
    resp = requests.post(API_ENDPOINT, json=payload)
    resp.raise_for_status()
    csv_text = resp.json()["result"]

    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    years = re.findall(r"(20\d{2})", csv_text)
    span = f"_{min(years)}{max(years)}" if years else ""

    path = raw_dir / f"{table_id.upper()}_{ts}{span}.csv"
    path.write_text(csv_text, encoding="utf-8")
    print(f"saved {path.name}")
    return path

# download_table("TEM12")

print("API helper defined.")
print(f"Endpoint  : {API_ENDPOINT}")
print(f"Raw dir   : {RAW_DIR}")
print()
print("To run a live download, uncomment the example above.")
print("Requires: pip install requests")

API helper defined.
Endpoint  : https://ws.cso.ie/public/api.jsonrpc
Raw dir   : /Users/ganeshsundararaman.arumugam/Desktop/Transport_latest/Transport_Decarbonisation_Dashboard_IE/data/raw

To run a live download, uncomment the example above.
Requires: pip install requests


## 4. Data freshness checker

Run below cell to inspect what is currently in `data/raw/` and whether every
expected table has a recent copy. A file is considered **stale** if it is older
than the number of days listed for its expected refresh cycle.

In [4]:
from datetime import datetime, timezone

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO_ROOT / "data" / "raw"

EXPECTED = {
    "TOA11": "Luas Passenger Numbers",
    "PEA01": "Population Estimates",
    "THA25": "PT Journeys (weekly)",
    "THA18": "Private Car Stock",
    "THA17": "All-Vehicle Stock & KM",
    "TEM12": "New Vehicles by Fuel Type",
    "TEM22": "Private Cars 2019-21 (archive)",
    "TEM23": "All Private Cars 2022+",
}

now = datetime.now(timezone.utc)
print(f"{'Table':<7} {'Label':<30} {'File':<25} Age")
for tid, label in EXPECTED.items():
    matches = sorted(RAW_DIR.glob(f"{tid}*.csv"))
    if not matches:
        print(f"{tid:<7} {label:<30} {'NOT FOUND':<25}")
        continue
    latest = matches[-1]
    age = (now - datetime.fromtimestamp(latest.stat().st_mtime, tz=timezone.utc)).days
    print(f"{tid:<7} {label:<30} {latest.name:<25} {age}d ago")

Table   Label                          File                      Age
TOA11   Luas Passenger Numbers         TOA11.20260528T000523 2018-2025.csv 12d ago
PEA01   Population Estimates           PEA01.20260528T000539 2018-2025.csv 12d ago
THA25   PT Journeys (weekly)           THA25.20260528T000529 2019-2025.csv 12d ago
THA18   Private Car Stock              THA18.20260528T000504 2018-2023.csv 12d ago
THA17   All-Vehicle Stock & KM         THA17.20260528T000501 2018-2023.csv 12d ago
TEM12   New Vehicles by Fuel Type      TEM12.20260527T230534 2015-2026.csv 12d ago
TEM22   Private Cars 2019-21 (archive) TEM22.20260528T010512 2019-2021.csv 12d ago
TEM23   All Private Cars 2022+         TEM23.20260528T000533 2022-2026.csv 12d ago


## 5. External reference data (`data/external/`)

The `data/external/` folder holds small hand-curated CSVs that the prescriptive
analytics notebook (notebook 04) reads as scenario parameters and policy targets.
These are **version-controlled** (unlike `data/raw/`) because they encode
deliberate analytical choices.

### Required files

| File | Contents | Source to verify |
|---|---|---|
| `policy_targets.csv` | Official 2030 targets by KPI | Climate Action Plan 2024, NTA Strategy |
| `scenario_params.csv` | Annual growth/adoption rates for each scenario | Project assumptions (document in METHODOLOGY.md) |
| `nat_population_projections.csv` | CSO M2F2 population projection to 2030 | CSO PEP01 |

### `policy_targets.csv` — starter template

```csv
kpi,target_year,target_value,target_unit,source,notes
car_dependency_index,2030,380,cars_per_1000,"CAP 2024 / OECD","Indicative; verify against latest CAP"
pt_usage_index,2030,90,journeys_per_capita,"NTA Strategy to 2030","Assumes doubling of 2019 baseline"
ev_phev_share,2030,0.50,proportion,"CAP 2024","30% of total fleet electric → ≈50% of new cars"
ev_phev_share,2030,0.45,proportion,"Conservative estimate","Based on current adoption trajectory"
```

> **Important:** Target values are indicative. Before finalising scenario 04,
> verify against the **current** Climate Action Plan (gov.ie/climateaction) and
> the NTA Sustainable Mobility Policy. The EPA (2025) projects Ireland will
> achieve only ~23% GHG reduction by 2030 vs the 51% target — the gap itself
> is a key analytical finding for the dashboard.

In [5]:
EXT_DIR = REPO_ROOT / "data" / "external"

print("Files needed before running notebook 04:")
for f in ["policy_targets.csv", "scenario_params.csv", "nat_population_projections.csv"]:
    exists = (EXT_DIR / f).exists()
    print(f, "found" if exists else "MISSING")

Files needed before running notebook 04:
policy_targets.csv MISSING
scenario_params.csv MISSING
nat_population_projections.csv MISSING


## 6. Update schedule and governance notes

| Dataset | Last significant revision | Next expected update | Action if updated |
|---|---|---|---|
| THA17 / THA18 | Annual, ~Q2 | Q2 2026 (2024 data) | Re-run notebook 01, check car dependency + intensity KPIs |
| TEM12 / TEM23 | Monthly | Each month | Re-run notebook 01 + 02 for latest fuel mix |
| THA25 | Weekly | Ongoing | Re-run notebook 01 + 02 for latest PT journeys |
| TOA11 | Monthly | Each month | Re-run notebook 01 + 02 for latest Luas numbers |
| PEA01 | Annual, ~August | August 2026 | Re-run all notebooks — population affects every per-capita KPI |
| TEM22 | Static archive | None | Do not re-download; already concatenated with TEM23 |

### What to do after a data refresh

1. Download the updated file(s) into `data/raw/` (section 2 or 3 above).
2. Run `01_data_preprocessing_and_cleaning.ipynb` → Kernel → Restart & Run All.
3. Check the `DATA_QUALITY_REPORT.md` audit log for any new warnings.
4. Re-run downstream notebooks (`02`, `03`, `04`) if KPI values change
   materially.
5. Commit the updated `data/processed/*.csv` and `docs/DATA_QUALITY_REPORT.md`.

> `data/raw/` is gitignored. Only `data/processed/` and `data/external/` are
> committed. This keeps the repository lean while ensuring the analysis-ready
> outputs are always reproducible.